In [1]:
import os
import re
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
import anndata as ad

# --- PATHS ---
WD = "/gpfs/commons/groups/knowles_lab/Karin/data/GTEx/v10"
GCT = f"{WD}/GTEx_subset_ATSE_overlap.gct.gz"                # rows=junctions, cols=samples
ATSE = f"{WD}/gtex_junctions_coordinates_atse_overlap.tsv"  # must contain 'junction_id'
SAMP_ATTR = f"{WD}/GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt"  # v8 SampleAttributes (GTEX- IDs)
IDS = None  # optional: path with one SAMPID per line

# ========================
# 0) Discover GCT columns
# ========================
cols = pd.read_csv(GCT, sep="\t", skiprows=2, nrows=0).columns.tolist()
cols[0] = cols[0].lstrip("\ufeff")  # strip BOM if present
name_col, desc_col = "Name", "Description"
sample_cols = [c for c in cols if c not in (name_col, desc_col)] if desc_col in cols else [c for c in cols if c != name_col]
if not sample_cols:
    raise RuntimeError("No sample columns discovered in GCT header.")
print(f"[GCT] header cols: {len(cols)} total (Name? {name_col in cols}, Description? {desc_col in cols})")
print(f"[GCT] sample_cols: {len(sample_cols)}; first 5: {sample_cols[:5]}")

# ===============================================
# 1) Build obs (GTEx-only) + optional IDS filter
# ===============================================
obs = (
    pd.read_csv(SAMP_ATTR, sep="\t", dtype=str, low_memory=False)
      .assign(SAMPID=lambda d: d["SAMPID"].str.replace(r"^\s*\ufeff", "", regex=True).str.strip())
      .set_index("SAMPID")
)
# Keep only GTEx rows
obs = obs.loc[[ix for ix in obs.index if ix.startswith("GTEX-")]]
print(f"[OBS] rows in SampleAttributes (GTEX only): {obs.shape[0]}")
print(f"[OBS] columns: {obs.shape[1]}; example cols: {list(obs.columns[:6])}")

# Optional: restrict to user-provided sample list
if IDS and os.path.exists(IDS):
    keep_ids = pd.read_csv(IDS, header=None, dtype=str).iloc[:, 0].dropna().astype(str).str.strip().tolist()
    obs = obs.loc[obs.index.intersection(keep_ids)]
    print(f"[OBS] after IDS filter: {obs.shape[0]} rows")

# Normalize sample IDs from GCT header the same way and intersect (preserve GCT order)
sample_cols = [re.sub(r"^\s*\ufeff", "", s).strip() for s in sample_cols]
sample_cols = [s for s in sample_cols if s in obs.index]
if not sample_cols:
    raise RuntimeError("No overlap between GCT samples and GTEx SampleAttributes v8.")
print(f"[ALIGN] overlap samples: {len(sample_cols)} (will stream). Example: {sample_cols[:3]}")

# ==========================================================
# 2) Stream GCT in chunks -> COO triplets (junctions × samples)
# ==========================================================
usecols_set = set([name_col] + sample_cols)
chunksize = 5000  # tune for RAM

data_vals, row_idx, col_idx = [], [], []
junction_ids = []
rows_seen = 0
nz_total = 0
chunk_no = 0

gct_iter = pd.read_csv(
    GCT,
    sep="\t",
    skiprows=2,
    usecols=lambda c: c in usecols_set,
    chunksize=chunksize,
    dtype={name_col: str},
    low_memory=False,
)

for chunk in gct_iter:
    chunk_no += 1
    jids = chunk[name_col].astype(str).to_numpy()
    vals = chunk[sample_cols].to_numpy(copy=False).astype(np.float32, copy=False)

    if chunk_no == 1:
        print(f"[STREAM] first chunk shape: {vals.shape} (rows=junctions, cols=samples)")
        print(f"[STREAM] first 3 junction_ids: {jids[:3]}")

    junction_ids.extend(jids)

    r, c = np.nonzero(vals)
    nnz = r.size
    nz_total += nnz
    if nnz:
        data_vals.extend(vals[r, c])
        row_idx.extend(rows_seen + r)
        col_idx.extend(c)

    rows_seen += vals.shape[0]
    if chunk_no % 50 == 0:
        print(f"[STREAM] chunk {chunk_no}: rows_seen={rows_seen:,} nnz_in_chunk={nnz:,} nnz_total={nz_total:,}")

n_rows = rows_seen
n_cols = len(sample_cols)
if n_rows == 0:
    raise RuntimeError("No rows were read from the GCT (unexpected).")
print(f"[STREAM DONE] junction rows read: {n_rows:,}; samples kept: {n_cols:,}")
print(f"[STREAM DONE] COO triplets: {len(data_vals):,} nonzeros")

# COO -> CSR and transpose to samples × junctions
M_coo = coo_matrix((data_vals, (row_idx, col_idx)), shape=(n_rows, n_cols), dtype=np.float32)
X = M_coo.tocsr().T
print(f"[MATRIX] X shape (samples × streamed junctions): {X.shape}")
print(f"[MATRIX] density: {X.nnz / (X.shape[0]*X.shape[1]):.6f}")

# ======================================
# 3) Align obs rows to matrix row order
# ======================================
obs = obs.loc[sample_cols]
obs.index.name = "SAMPID"

# ===========================================
# 4) Build var from ATSE and align columns
# ===========================================
var = pd.read_csv(ATSE, sep="\t", dtype=str)
if "junction_id" not in var.columns:
    raise RuntimeError("'junction_id' column not found in ATSE table.")
var = var.drop_duplicates(subset=["junction_id"]).set_index("junction_id")
print(f"[VAR] ATSE rows: {var.shape[0]:,}; cols: {var.shape[1]}")
print(f"[VAR] first 3 ATSE junction_ids: {var.index[:3].tolist()}")
print(f"[VAR] streamed junction_ids example: {junction_ids[:3]}")

junction_index = pd.Index(junction_ids, name="junction_id")
overlap = junction_index.intersection(var["original_junction_id"])
if overlap.size == 0:
    raise RuntimeError("No overlap between streamed junction IDs and ATSE['junction_id'].")

pos_map = pd.Series(np.arange(junction_index.size), index=junction_index)
keep_pos = pos_map.loc[overlap].to_numpy()

X = X[:, keep_pos]

print(f"[ALIGN VAR] overlap junctions: {overlap.size:,} (from streamed {len(junction_ids):,} ∩ ATSE {var.shape[0]:,})")
print(f"[ALIGN VAR] X after column subset: {X.shape}")

# =====================
# 5) Build AnnData
# =====================
adata = ad.AnnData(X=X.tocsr(), obs=obs.copy(), var=var.copy())
adata.obs_names.name = "SAMPID"
adata.var_names.name = "junction_id"
adata.layers["counts"] = adata.X.copy()

# QC metrics
adata.obs["n_counts"] = np.asarray(adata.X.sum(axis=1)).ravel()
adata.obs["n_junctions_detected"] = np.asarray((adata.X > 0).sum(axis=1)).ravel()

print(f"[AnnData] obs: {adata.obs.shape}; var: {adata.var.shape}; X: {adata.X.shape}")
print(f"[AnnData] nnz: {adata.X.nnz:,} density={adata.X.nnz/(adata.n_obs*adata.n_vars):.6f}")
print("[AnnData] example obs index:", adata.obs_names[:3].tolist())
print("[AnnData] example var index:", adata.var_names[:3].tolist())
print(adata)

[GCT] header cols: 19790 total (Name? True, Description? True)
[GCT] sample_cols: 19788; first 5: ['GTEX-1117F-0005-SM-HL9SH', 'GTEX-1117F-0011-R10b-SM-GI4VE', 'GTEX-1117F-0011-R11b-SM-GIN8R', 'GTEX-1117F-0011-R2b-SM-GI4VL', 'GTEX-1117F-0011-R3a-SM-GJ3PJ']
[OBS] rows in SampleAttributes (GTEX only): 22734
[OBS] columns: 62; example cols: ['SMATSSCR', 'SMCENTER', 'SMPTHNTS', 'SMRIN', 'SMTS', 'SMTSD']
[ALIGN] overlap samples: 16622 (will stream). Example: ['GTEX-1117F-0226-SM-5GZZ7', 'GTEX-1117F-0426-SM-5EGHI', 'GTEX-1117F-0526-SM-5EGHJ']
[STREAM] first chunk shape: (5000, 16622) (rows=junctions, cols=samples)
[STREAM] first 3 junction_ids: ['chr1:827776-829002:+' 'chr1:827776-841199:+' 'chr1:827850-829002:+']
[STREAM DONE] junction rows read: 106,206; samples kept: 16,622
[STREAM DONE] COO triplets: 1,252,945,571 nonzeros
[MATRIX] X shape (samples × streamed junctions): (16622, 106206)
[MATRIX] density: 0.709741
[VAR] ATSE rows: 106,206; cols: 6
[VAR] first 3 ATSE junction_ids: ['chr1_8

In [11]:
# =====================
# 6) Debug and clean data types, then save AnnData object
# =====================

# First, let's inspect the problematic column
print("[DEBUG] Inspecting SMGTC column:")
print(f"  dtype: {adata.obs['SMGTC'].dtype}")
print(f"  unique values: {adata.obs['SMGTC'].value_counts()}")
print(f"  sample values: {adata.obs['SMGTC'].iloc[:5].tolist()}")
print(f"  has NaN: {adata.obs['SMGTC'].isna().any()}")

# More aggressive cleaning approach
print("\n[CLEANING] Converting all obs columns to categorical or string...")
for col in adata.obs.columns:
    if col in ['n_counts', 'n_junctions_detected']:
        # Keep numeric QC columns as-is
        continue
    
    print(f"Processing column: {col}")
    
    # Convert to string first, handling all edge cases
    series = adata.obs[col].copy()
    
    # Handle different data types
    if series.dtype == 'object':
        # Convert everything to string, replacing problematic values
        series = series.astype(str)
        series = series.replace(['nan', 'None', 'NaN', '<NA>'], 'Unknown')
        
        # Convert to categorical to save space and avoid string issues
        adata.obs[col] = pd.Categorical(series)
        print(f"  -> converted to categorical (n_categories: {len(adata.obs[col].cat.categories)})")
    
    elif pd.api.types.is_numeric_dtype(series):
        # Keep numeric but ensure no NaN issues
        if series.isna().any():
            series = series.fillna(-1)  # or some appropriate fill value
        adata.obs[col] = series
        print(f"  -> kept as numeric")

# Clean var DataFrame too
print("\n[CLEANING] Converting var columns...")
for col in adata.var.columns:
    if adata.var[col].dtype == 'object':
        series = adata.var[col].astype(str).replace(['nan', 'None', 'NaN'], 'Unknown')
        adata.var[col] = pd.Categorical(series)
        print(f"  -> {col}: converted to categorical")

# Define output path
output_path = f"{WD}/gtex_junctions_anndata.h5ad"

print(f"\n[SAVING] Attempting to save to {output_path}...")
try:
    # Try with LZF first
    adata.write(output_path, compression='lzf')
    print(f"[SUCCESS] AnnData object saved with LZF compression")
except Exception as e:
    print(f"[WARNING] LZF failed: {e}")
    try:
        # Fallback to gzip
        adata.write(output_path, compression='gzip')
        print(f"[SUCCESS] AnnData object saved with gzip compression")
    except Exception as e2:
        print(f"[WARNING] gzip failed: {e2}")
        # Last resort: no compression
        adata.write(output_path, compression=None)
        print(f"[SUCCESS] AnnData object saved without compression")

# Quick verification
print(f"[VERIFY] File size: {os.path.getsize(output_path) / 1e9:.2f} GB")

# Optional: save summary statistics
summary_stats = {
    'n_samples': adata.n_obs,
    'n_junctions': adata.n_vars, 
    'n_nonzero': adata.X.nnz,
    'density': adata.X.nnz / (adata.n_obs * adata.n_vars),
    'mean_counts_per_sample': adata.obs['n_counts'].mean(),
    'mean_junctions_per_sample': adata.obs['n_junctions_detected'].mean()
}

summary_df = pd.DataFrame([summary_stats])
summary_df.to_csv(f"{WD}/gtex_junctions_summary.csv", index=False)
print("[SAVED] Summary statistics saved to gtex_junctions_summary.csv")

[DEBUG] Inspecting SMGTC column:
  dtype: object
  unique values: Series([], Name: count, dtype: int64)
  sample values: [nan, nan, nan, nan, nan]
  has NaN: True

[CLEANING] Converting all obs columns to categorical or string...
Processing column: SMATSSCR
Processing column: SMCENTER
Processing column: SMPTHNTS
Processing column: SMRIN
Processing column: SMTS
Processing column: SMTSD
Processing column: SMUBRID
Processing column: SMTSISCH
Processing column: SMTSPAX
Processing column: SMNABTCH
Processing column: SMNABTCHT
Processing column: SMNABTCHD
Processing column: SMGEBTCH
Processing column: SMGEBTCHD
Processing column: SMGEBTCHT
Processing column: SMAFRZE
Processing column: SMGTC
  -> converted to categorical (n_categories: 1)
Processing column: SME2MPRT
Processing column: SMCHMPRS
Processing column: SMNTRART
Processing column: SMNUMGPS
  -> converted to categorical (n_categories: 1)
Processing column: SMMAPRT
Processing column: SMEXNCRT
Processing column: SM550NRM
  -> converted 